# ✅ Flujo 3: Calidad de datos
### Audita antes de fiarte

Nulos, duplicados y rangos
raros. Nunca confíes
en los datos a ciegas.

In [ ]:
from pyspark.sql import (
    SparkSession, functions as F)

spark = (
    SparkSession.builder
    .appName('calidad_datos')
    .getOrCreate()
)

pedidos = spark.createDataFrame([
  (1, 'Ana',   120, 25),
  (2, 'Luis',  None, 40),
  (3, None,    80, 33),
  (4, 'Sara', -5, 29),
  (2, 'Luis',  200, 40),
  (5, 'Pol',   90, 200),
], ['id','cliente','importe','edad'])

pedidos.show()

## 1️⃣ Nulos por columna
### ¿Dónde faltan datos?

In [ ]:
nulos = pedidos.select([
  F.count(
    F.when(F.col(c).isNull(), c)
  ).alias(c)
  for c in pedidos.columns
])

nulos.show()

## 2️⃣ Claves duplicadas
### El id debería ser único

In [ ]:
duplicados = (
  pedidos
  .groupBy('id')
  .count()
  .filter(F.col('count') > 1)
)

print("IDs duplicados:")
duplicados.show()

## 3️⃣ Fuera de rango
### Importe < 0, edad > 120

Señales de datos corruptos.

In [ ]:
sospechosos = pedidos.filter(
  (F.col('importe') < 0)
  | (F.col('edad') < 0)
  | (F.col('edad') > 120)
)

print("Filas sospechosas:")
sospechosos.show()

## 4️⃣ Informe de calidad

In [ ]:
total = pedidos.count()
unicos = pedidos.select('id').distinct().count()

print(f"Filas totales:     {total}")
print(f"IDs únicos:        {unicos}")
print(f"Duplicados:        {total - unicos}")
print(f"Filas sospechosas: {sospechosos.count()}")

spark.stop()